In [19]:
import pandas as pd
import langextract as lx
from rich.pretty import pprint
import textwrap

https://github.com/google/langextract/tree/main/examples/ollama 

# New prompt

In [29]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- Tené en cuenta que, si bien las fechas sensibles al caso deben sacarse, la única fecha que debe permanecer es la fecha de resolución que suele anunciarse al comienzo como
"Buenos Aires, ... "
- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, extráela igual.
- Si una entidad está incompleta, extráela igual.
- Si una entidad está repetida, extráelas todas.
- Si una entidad está en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Entidad bancaria
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura, intersección de calles, o número de domicilio
- DNI: documento nacional de identidad (7-8 dígitos)
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares)
- LINK: URLs o enlaces web
- LOC: nombres de localidades, provincias, países, continentes
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).
                         
Razona detenidamente antes de elegir casa entidad y asociá dicho razonamiento a una variable justification y score de confianza de tu razonamiento, luego, para todo el parrafo crea una 
salida que sea una lista de diccionarios, uno por clase extraida, con las siguientes claves:
- text: el texto exacto de la entidad
- label: la clase de la entidad (una de las listadas más arriba)
                 
The output must be a JSON array of objects, one per entity, like this:
[
  {"text": "Banco Nación",
   "label": "BANCO"},

  {"text": "2850590940090418135201",
   "start_char": 138,
   "end_char": 160,
   "label": "CBU",
   "justification": "Número de 22 dígitos con formato válido de CBU.",
   "score": 0.99,
   "model": "openai/gpt-4"},

  {"text": "ejemplo@gmail.com",
   "start_char": 469,
   "end_char": 486,
   "label": "CORREO_ELECTRONICO",
   "justification": "Dirección con formato de email estándar.",
   "score": 0.99,
   "model": "openai/gpt-4"},
""")


In [ ]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, "
              "el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle "
              "Sarmiento 1234, localidad de Moreno."""),
        extractions=[
            lx.data.Extraction(extraction_class="FECHA",     extraction_text="5 de mayo de 2023"),
            lx.data.Extraction(extraction_class="PER",       extraction_text="Juan Pérez"),
            lx.data.Extraction(extraction_class="DIRECCION", extraction_text="Sarmiento 1234"),
            lx.data.Extraction(extraction_class="LOC",       extraction_text="Moreno"),
        ],
    ),

    lx.data.ExampleData(
        text = textwrap.dedent(""""3) Abstenerse de ingresar y/o concurrir a la Villa 13"""),
        extractions = [
            lx.data.Extraction(extraction_class="LOC", extraction_text = "Villa 13")
        ]
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""JUZGADO NACIONAL EN LO CRIMINAL Y CORRECCIONAL N° 10 - Secretaría N° 19. "
              "Causa N° 52345/2022. CUIJ: 12-34567890-1. Actuación N° 2022-009876."""),
        extractions=[
            lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"),
            lx.data.Extraction(extraction_class="CUIJ",           extraction_text="12-34567890-1"),
            lx.data.Extraction(extraction_class="NUM_ACTUACION",  extraction_text="2022-009876"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Comparece Miguel Torres, DNI 30123456, de 34 años de edad, nacionalidad paraguaya, "
              "con estudios secundarios completos, con último domicilio en Av. Corrientes 3456 de esta ciudad, "
              "junto a su cuñado Jorge Pérez."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="30123456"),
            lx.data.Extraction(extraction_class="EDAD",        extraction_text="34"),
            lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya"),
            lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos"),
            lx.data.Extraction(extraction_class="DIRECCION",   extraction_text="Av. Corrientes 3456"),
            lx.data.Extraction(extraction_class="PER",         extraction_text="Jorge Pérez"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""1) transferencia defondos a cuenta de terceros por $167.000,- hacia una cuenta a nombre de la Sra. Carla Analía Gonzales, CUIL 27-25011757-0, CBU 0740399088000036512321, del Banco Santander. """),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Analía Gonzales"),
            lx.data.Extraction(extraction_class="CUIL",       extraction_text="27-25011757-0"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0740399088000036512321"),
            lx.data.Extraction(extraction_class="BANCO",      extraction_text="Banco Santander"),
        ],
    ),

  lx.data.ExampleData(
        text=textwrap.dedent("""teléfono celular 1141504528 y dirección de correo electrónico alejandro.perezgarcia@gmail.com."""),
        extractions=[
            lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528"),
            lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="alejandro.perezgarcia@gmail.com"),
        ],
    ),
lx.data.ExampleData(
        text=textwrap.dedent("""Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a  ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876 """),
   extractions=[
            lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"),
            lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Cecilia Lopez Gracia médica del Hospital Penna, Servicio SAME, M. N. 123.558."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Cecilia Lopez Gracia"),
            lx.data.Extraction(extraction_class="NUM_MATRICULA", extraction_text="M. N. 123.558"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""A  su  vez,  requirió  informes  al  Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  de  la  denunciante,  Carla Alejandra Garcia, D.N.I.  36.998.621 identificadas  como  Caja  de  ahorro  en  pesos  argentinos  número  117-59824/6  con  CBU  0180132640000004685591 y  Caja  de  ahorro  en  dólares  número  119-619018/2 con  CBU  0170115544000062081822."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Alejandra Garcia"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="36.998.621"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0170115544000062081822"),
        ],
    ), 

    lx.data.ExampleData(
        text=textwrap.dedent("""La grabación se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"""),
        extractions=[
            lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

In [22]:
print("""A O
su O
vez, O
requirió O
informes O
al O
Banco B-BANCO
BBVA I-BANCO
Francés, I-BANCO
respecto O
de O
las O
cuentas O
bancarias O
de O
la O
denunciante, O
Florencia B-PER
Alejandra I-PER
Follino, I-PER
D.N.I. O
33.085.629 B-DNI
identificadas O
como O
Caja O
de O
ahorro O
en O
pesos O
argentinos O
número O
118-46824/6 B-NUM_CAJA_AHORRO
con O
CBU O
0170118640000004682468 B-CBU
y O
Caja O
de O
ahorro O
en O
dólares O
número O
118-618818/2 B-NUM_CAJA_AHORRO
con O
CBU O
0170118644000061881822.""".replace("O\n", " ").replace("I-PER","").replace("I-BANCO","").replace("I-CBU","").replace("I-NUM_CAJA_AHORRO","").replace("B-DNI","").replace("\n","")    )   

A  su  vez,  requirió  informes  al  Banco B-BANC BBVA I-BANC Francés, I-BANC respecto  de  las  cuentas  bancarias  de  la  denunciante,  Florencia B-PERAlejandra Follino, D.N.I.  33.085.629 identificadas  como  Caja  de  ahorro  en  pesos  argentinos  número  118-46824/6 B-NUM_CAJA_AHORR con  CBU  0170118640000004682468 B-CBUy  Caja  de  ahorro  en  dólares  número  118-618818/2 B-NUM_CAJA_AHORR con  CBU  0170118644000061881822.


In [31]:
text = ("La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960).")

In [13]:
text = ("El día 28 de diciembre de 2021 llevé adelante el debate oral y público, en el cual juzgué a JUAN CARLOS GOMEZ, asistido por el Defensor Oficial Juan Ignacio Cafiero, titular de la Defensoría 18; según los hechos atribuidos por la querellante SUSANA ALVAREZ, asistida por su abogada Daniela Ribas (Tº 130 Fº 110 CPACF), consistentes en:  Haberse sustraído de prestar los medios indispensables para la subsistencia de sus hijos menores de edad S.B.A. desde el mes de octubre de 2016 hasta diciembre de 2019. Dicha conducta fue encuadrada en la figura de incumplimiento de los deberes de asistencia familiar, prevista y reprimida por el art. 1 de la Ley 13.944. Al inicio del debate la letrada de la querellante adelantó que había decidido ampliar el período de la imputación respecto de S.B.A  hasta ese día, es decir 28 de diciembre de 2021, y respecto de la adolescente S.B.A hasta el mes de marzo del 2021, fecha en la que cumplió la mayoría de edad. En atención a ello, y en orden a lo establecido en el art. 242 del CPP consulté a la defensa acerca de la necesidad o no de contar con un plazo para poder prepararse.")

In [7]:
df = pd.read_csv('documents-02-08-sin05.csv')

In [ ]:
text = df.text[20]

In [15]:
text = df.text[124]
text

'Imputado: quiere vivir en la calle Maza 1246, PB, dpto. "2", de esta Ciudad, donde residiría con su actual pareja.'

### OpenAI

In [32]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

result = lx.extract(
    text_or_documents=text,
    prompt_description=PROMPT,
    examples=examples,
    language_model_type=lx.inference.OpenAILanguageModel,
    model_id="gpt-4o",
    api_key=openai_api_key,
    max_char_buffer=1000,
    extraction_passes=1,
    max_workers=6,
    fence_output=True,
    use_schema_constraints=False, # https://github.com/google/langextract
    language_model_params={
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 400,
        "timeout": 600,
    },
)

/var/folders/yf/2cttyjmx5j35yhq4st33mdmm0000gn/T/ipykernel_5118/2065202320.py:7: DeprecationWarning: 'language_model_type' is deprecated and will be removed in v2.0.0. Use model, config, or model_id parameters instead.
  result = lx.extract(
2025-08-27 15:26:01,290 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.__init__(self=<OpenAILanguageModel>, constraint=Constraint(co...NONE: 'none'>), kwargs={})
2025-08-27 15:26:01,292 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.__init__ -> None (0.0 ms)
2025-08-27 15:26:01,295 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.apply_schema(self=<OpenAILanguageModel>, schema_instance=None)
2025-08-27 15:26:01,296 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.apply_schema -> None (0.0 ms)
DEBUG:absl:Initialized Annotator with prompt:

Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en es

✓ Extraction processing complete


INFO:absl:Finalizing annotation for document ID doc_1677311a.
INFO:absl:Document annotation completed.


✓ Extracted 4 entities (2 unique types)
  • Time: 7.49s
  • Speed: 39 chars/sec
  • Chunks: 1


In [33]:
result

AnnotatedDocument(extractions=[Extraction(extraction_class='FECHA', extraction_text='12 de marzo de 2023', char_interval=CharInterval(start_pos=94, end_pos=113), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={}), Extraction(extraction_class='PER', extraction_text='Carlos Gómez', char_interval=CharInterval(start_pos=163, end_pos=175), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={}), Extraction(extraction_class='PER', extraction_text='María Rodriguez', char_interval=CharInterval(start_pos=178, end_pos=193), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=2, description=None, attributes={}), Extraction(extraction_class='PER', extraction_text='Juan Pérez', char_interval=CharInterval(start_pos=206, end_pos=216), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extractio

### LLama3.2

In [4]:
'''
result = lx.extract(
    text_or_documents=text,
    prompt_description=PROMPT,
    examples=examples,
    language_model_type=lx.inference.OllamaLanguageModel,
    model_id="llama3.2:3b",
    model_url="http://host.docker.internal:11434",
    max_char_buffer=1000,
    extraction_passes=1,
    max_workers=6,
    fence_output=False,
    use_schema_constraints=False,
    language_model_params={
        "temperature": 0.1,          # menos creatividad - > checkear que con t=0 sea determinista
        "top_p": 0.9,
        "top_k": 40,
        "max_output_tokens": 400,
        "timeout": 600,
        "keep_alive": "10m",
        "num_ctx": 4096,             # cuántos tokens de contexto
    },
)
'''

'\nresult = lx.extract(\n    text_or_documents=text,\n    prompt_description=PROMPT,\n    examples=examples,\n    language_model_type=lx.inference.OllamaLanguageModel,\n    model_id="llama3.2:3b",\n    model_url="http://host.docker.internal:11434",\n    max_char_buffer=1000,\n    extraction_passes=1,\n    max_workers=6,\n    fence_output=False,\n    use_schema_constraints=False,\n    language_model_params={\n        "temperature": 0.1,          # menos creatividad - > checkear que con t=0 sea determinista\n        "top_p": 0.9,\n        "top_k": 40,\n        "max_output_tokens": 400,\n        "timeout": 600,\n        "keep_alive": "10m",\n        "num_ctx": 4096,             # cuántos tokens de contexto\n    },\n)\n'

In [18]:
print(f"Extracted {len(result.extractions)} entities from {len(result.text):,} characters")

# Save and visualize the results
lx.io.save_annotated_documents([result], output_name="test2_extractions.jsonl", output_dir=".")

# Generate the interactive visualization
html_content = lx.visualize("test2_extractions.jsonl")
with open("test_extractions.html", "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

print("Interactive visualization saved to test_extractions.html")

Extracted 7 entities from 114 characters


LangExtract: Saving to test2_extractions.jsonl: 1 docs [00:00, 389.59 docs/s]

✓ Saved 1 documents to test2_extractions.jsonl



LangExtract: Loading test2_extractions.jsonl: 100%|██████████| 1.66k/1.66k [00:00<00:00, 1.01MB/s]

✓ Loaded 1 documents from test2_extractions.jsonl
Interactive visualization saved to test_extractions.html


In [17]:

# Analyze character mentions
characters = {}
for e in result.extractions:
    if e.extraction_class == "character":
        char_name = e.extraction_text
        if char_name not in characters:
            characters[char_name] = {"count": 0, "attributes": set()}
        characters[char_name]["count"] += 1
        if e.attributes:
            for attr_key, attr_val in e.attributes.items():
                characters[char_name]["attributes"].add(f"{attr_key}: {attr_val}")

# Print character summary
print(f"\nCHARACTER SUMMARY ({len(characters)} unique characters)")
print("=" * 60)

sorted_chars = sorted(characters.items(), key=lambda x: x[1]["count"], reverse=True)
for char_name, char_data in sorted_chars[:10]:  # Top 10 characters
    attrs_preview = list(char_data["attributes"])[:3]
    attrs_str = f" ({', '.join(attrs_preview)})" if attrs_preview else ""
    print(f"{char_name}: {char_data['count']} mentions{attrs_str}")

# Entity type breakdown
entity_counts = Counter(e.extraction_class for e in result.extractions)
print(f"\nENTITY TYPE BREAKDOWN")
print("=" * 60)
for entity_type, count in entity_counts.most_common():
    percentage = (count / len(result.extractions)) * 100
    print(f"{entity_type}: {count} ({percentage:.1f}%)")


CHARACTER SUMMARY (0 unique characters)


NameError: name 'Counter' is not defined

In [ ]:
import re
from langextract.data import AlignmentStatus

PATTERNS = {
    "CUIJ": r"\b\d{2}-\d{8}-\d\b",
    "NUM_EXPEDIENTE": r"\b\d{1,8}/\d{4}\b",
    "NUM_ACTUACION": r"\b\d{4}[-/]\d{4,10}\b",
    "CBU": r"\b\d{22}\b",
    "CUIT_CUIL": r"\b\d{2}-\d{8}-\d\b",
    "DNI": r"\b\d{7,8}\b",
    "PATENTE_DOMINIO": r"\b([A-Z]{3}\d{3}|[A-Z]{2}\d{3}[A-Z]{2})\b",
    "CORREO_ELECTRONICO": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "LINK": r"\bhttps?://\S+\b",
}

def validate_by_regex(cls, text):
    pat = PATTERNS.get(cls)
    return True if not pat else re.search(pat, text) is not None

def filter_extractions(doc):
    base = doc.text
    seen = set()
    kept = []
    for e in doc.extractions:
        # 1) Debe venir con grounding
        if not e.char_interval:
            continue
        s, t = e.char_interval.start_pos, e.char_interval.end_pos
        if s is None or t is None or s < 0 or t > len(base):
            continue
        span = base[s:t]
        # 2) Debe coincidir EXACTO
        if span != e.extraction_text:
            continue
        # 3) Regex para clases numéricas
        if not validate_by_regex(e.extraction_class, e.extraction_text):
            continue
        # 4) Dedupe
        key = (e.extraction_class, e.extraction_text, s, t)
        if key in seen:
            continue
        seen.add(key)
        kept.append(e)
    doc.extractions = kept
    return doc

result = filter_extractions(output)


# Old

In [4]:
# 1. Define the prompt and extraction rules
prompt = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [15]:
prompt = textwrap.dedent("""
Eres un asistente que extrae ENTIDADES sensibles para anonimización en documentos judiciales en español.
Reglas:
- Usa TEXTO EXACTO del documento (no parafrasees).
- No superpongas entidades; una mención = una extracción.
- Si tenés dudas, no inventes.
Clases permitidas y guía breve:
- BANCO: nombre de entidad bancaria.
- CBU: 22 dígitos continuos.
- CORREO_ELECTRONICO: formato correo válido.
- CUIJ: código causa judicial (ej. 12-34567890-1).
- CUIT_CUIL: CUIT/CUIL (##-########-#).
- DIRECCION: calle y número (opcionalmente ciudad/barrio).
- DNI: número de documento (solo dígitos).
- EDAD: número de edad explícito (en años).
- ESTUDIOS: nivel/institución educativa que identifique a la persona.
- FECHA: dd/mm/aaaa, dd-mm-aaaa o “5 de mayo de 2023”.
- LINK: URL http/https.
- LOC: localidad/barrio/comisaría/ciudad.
- MARCA_AUTOMOVIL: marca (Toyota, Ford, Volkswagen, etc.).
- NACIONALIDAD: gentilicio (argentino, paraguaya, ...).
- NUM_ACTUACION: nro. de actuación administrativa/contravencional.
- NUM_CAJA_AHORRO: número de caja de ahorro/cuenta.
- NUM_EXPEDIENTE: nro. de expediente.
- NUM_MATRICULA: matrícula profesional o académica.
- PATENTE_DOMINIO: dominio vehicular (p. ej., AB123CD).
- PER: nombre(s) y apellido(s) de persona física. También apodos/iniciales.

Salida: solo las entidades que estén explícitas en el texto.
""").strip()

In [17]:
# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan Pérez"
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(
                extraction_class="LOC", extraction_text="Moreno"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carlos Gómez"
            ),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Miguel Torres"
            ),
            lx.data.Extraction(
                extraction_class="DNI", extraction_text="30123456"
            ),
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="14/02/1990"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Jorge Pérez"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan López"
            ),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Ana García"
            ),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
]

In [18]:
text = "La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960)."

# Run the extraction
result = lx.extract(
    text_or_documents=text,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.2:3b",
    model_url="http://host.docker.internal:11434",
    max_workers=20,
    fence_output=False,
    language_model_params={
            "timeout": 600,       # aumentar el timeout a 10 minutos
            "keep_alive": "5m"    # mantener modelo cargado 5 minutos
        },
    use_schema_constraints=False,
)

/workspace/.venv/lib/python3.10/site-packages/langextract/__init__.py:186: UserWarning: batch_length (10) < max_workers (20). Only 10 workers will be used. Set batch_length >= max_workers for optimal parallelization.
  warnings.warn(
2025-08-22 17:30:53,380 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.__init__(self=<OllamaLanguageModel>, constraint=Constraint(co...NONE: 'none'>), kwargs={})
2025-08-22 17:30:53,381 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.__init__ -> None (0.0 ms)
2025-08-22 17:30:53,382 - langextract.debug - DEBUG - [langextract.inference] CALL: BaseLanguageModel.apply_schema(self=<OllamaLanguageModel>, schema_instance=None)
2025-08-22 17:30:53,382 - langextract.debug - DEBUG - [langextract.inference] RETURN: BaseLanguageModel.apply_schema -> None (0.0 ms)
DEBUG:absl:Initialized Annotator with prompt:
Eres un asistente que extrae ENTIDADES sensibles para anonimización en documentos judiciales en es

✓ Extraction processing complete



INFO:absl:Finalizing annotation for document ID doc_093bbb5b.
INFO:absl:Document annotation completed.


✓ Extracted 8 entities (7 unique types)
  • Time: 338.25s
  • Speed: 1 chars/sec
  • Chunks: 1


In [22]:
print(text)

La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960).


In [19]:
pprint(result)

AnnotatedDocument(
│   extractions=[
│   │   Extraction(
│   │   │   extraction_class='CUIJ',
│   │   │   extraction_text='12-34567890-1',
│   │   │   char_interval=None,
│   │   │   alignment_status=None,
│   │   │   extraction_index=1,
│   │   │   group_index=0,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='NUM_EXPEDIENTE',
│   │   │   extraction_text='52345/2022',
│   │   │   char_interval=None,
│   │   │   alignment_status=None,
│   │   │   extraction_index=2,
│   │   │   group_index=1,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='DNI',
│   │   │   extraction_text='30123456',
│   │   │   char_interval=None,
│   │   │   alignment_status=None,
│   │   │   extraction_index=3,
│   │   │   group_index=2,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='Carlos Gómez',
│   │   │   char_interval=CharInterval(start_pos=163, end_pos=175),
│   │   │   alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>,
│   │   │   extraction_index=4,
│   │   │   group_index=3,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='FECHA',
│   │   │   extraction_text='12 de marzo de 2023',
│   │   │   char_interval=CharInterval(start_pos=94, end_pos=113),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=5,
│   │   │   group_index=4,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='NACIONALIDAD',
│   │   │   extraction_text='argentina',
│   │   │   char_interval=None,
│   │   │   alignment_status=None,
│   │   │   extraction_index=6,
│   │   │   group_index=5,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='PER',
│   │   │   extraction_text='María Rodriguez',
│   │   │   char_interval=CharInterval(start_pos=178, end_pos=193),
│   │   │   alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>,
│   │   │   extraction_index=7,
│   │   │   group_index=6,
│   │   │   description=None,
│   │   │   attributes={}
│   │   ),
│   │   Extraction(
│   │   │   extraction_class='NUM_ACTUACION',
│   │   │   extraction_text='2022-009876',
│   │   │   char_interval=None,
│   │   │   alignment_status=None,
│   │   │   extraction_index=8,
│   │   │   group_index=7,
│   │   │   description=None,
│   │   │   attributes={}
│   │   )
│   ],
│   text='La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960).'
)